# Feature engineering

I build pre-match features for each team. Every feature uses only matches played before the one I predict.

## 1. Recent form
For each team and each match, I compute the average points over the 5 previous matches and the goals scored and conceded over the 10 previous matches.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.database import query

In [5]:
query("""
    SELECT m.date, m.opponent, m.goals_for, m.goals_against,
           f.n_previous, f.points_last_5, f.goals_for_last_10, f.goals_against_last_10
    FROM team_matches AS m
    JOIN team_form AS f USING (match_id, team)
    WHERE m.team = 'France'
    ORDER BY m.date DESC
    LIMIT 8
""")

,date,opponent,goals_for,goals_against,n_previous,points_last_5,goals_for_last_10,goals_against_last_10
0,2026-07-18,England,4,6,939,2.4,2.3,0.8
1,2026-07-14,Spain,0,2,938,3.0,2.5,0.7
2,2026-07-09,Morocco,2,0,937,3.0,2.6,0.8
3,2026-07-04,Paraguay,1,0,936,3.0,2.9,0.8
4,2026-06-30,Sweden,3,0,935,2.4,2.8,1.0
5,2026-06-26,Norway,4,1,934,2.4,2.7,0.9
6,2026-06-22,Iraq,3,0,933,2.4,2.6,1.0
7,2026-06-16,Senegal,3,1,932,2.4,2.5,0.9


Before the semi-final against Spain, France had won its 5 previous matches, so `points_last_5` is 3.0.
The loss against Spain is not in its own row. It only enters the next row: (0 + 3 + 3 + 3 + 3) / 5 = 2.4. The window stops before the current match, so there is no leakage.

In [3]:
query("""
    SELECT COUNT(*) AS first_matches, COUNT(points_last_5) AS form_not_null
    FROM team_form
    WHERE n_previous = 0
""")

,first_matches,form_not_null
0,224,0


In [4]:
query("""
    SELECT COUNT(*) AS team_rows,
           ROUND(AVG((f.n_previous >= 10)::INTEGER), 3) AS share_10_previous
    FROM team_form AS f
    JOIN clean_results AS c USING (match_id)
    WHERE c.date >= '2000-01-01'
""")

,team_rows,share_10_previous
0,47920,0.999


A team's first match has no form (NULL), as expected: nothing is known before it.
Since 2000, 99.9% of team rows have at least 10 previous matches, so the form features are almost always computed on a full window.
128 times a team played two matches on the same day (split squads in friendlies). The second match sees the first result. This is marginal and I note it as a limitation.